In [2]:
import undetected_chromedriver as uc
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait               # ---> NEW IMPORT
from selenium.webdriver.support import expected_conditions as EC      # ---> NEW IMPORT
from selenium.webdriver.common.action_chains import ActionChains      # ---> NEW IMPORT

import numpy as np
import ast
import pandas as pd
import openpyxl
import os

# ---> 1. PASTE THE HELPER FUNCTION HERE AT THE TOP <--
def handle_cloudflare(driver, url):
    driver.get(url)
    try:
        print("Checking for Cloudflare Turnstile...")
        iframe = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//iframe[contains(@title, 'Cloudflare security challenge')]"))
        )
        print("Cloudflare detected! Attempting to mimic human click...")
        time.sleep(3.5)
        ActionChains(driver).move_to_element(iframe).click(iframe).perform()
        time.sleep(5)
    except Exception as e:
        print("No turnstile detected (or timed out). Proceeding with scraping!")
    return driver


options = uc.ChromeOptions()
# ... (your normal options) ...

if os.environ.get("CI"):
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    
    driver = uc.Chrome(
        options=options,
        version_main=146,
        browser_executable_path="/opt/chrome146/chrome",
        driver_executable_path="/usr/local/bin/chromedriver146",
        use_subprocess=False    
    )
    print("✅ Driver started")
else:
    driver = uc.Chrome(options=options)


# ---> 2. USE THE FUNCTION INSTEAD OF driver.get() <---
# Wherever you used to just do `driver.get('https://partsouq.com')`,
# replace that single line with a call to the new function! Like this:

print("Opening Partsouq and checking for Turnstile...")
driver = handle_cloudflare(driver, 'https://partsouq.com')

# Make sure it actually opens it
# Take a screenshot to verify!
driver.save_screenshot('output/Hyundai_screenshot2.png')

SessionNotCreatedException: Message: session not created: cannot connect to chrome at 127.0.0.1:53951
from session not created: This version of ChromeDriver only supports Chrome version 147
Current browser version is 146.0.7680.165; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
0   undetected_chromedriver             0x0000000100a7fd38 undetected_chromedriver + 6798648
1   undetected_chromedriver             0x0000000100a773ba undetected_chromedriver + 6763450
2   undetected_chromedriver             0x0000000100472825 undetected_chromedriver + 452645
3   undetected_chromedriver             0x00000001004b6049 undetected_chromedriver + 729161
4   undetected_chromedriver             0x00000001004b4e51 undetected_chromedriver + 724561
5   undetected_chromedriver             0x00000001004a9b89 undetected_chromedriver + 678793
6   undetected_chromedriver             0x00000001004fb

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import time
import pandas as pd
import numpy as np

Model = ['Hyundai', 'Kia']
data = []

for i in Model:
    url = "https://partsouq.com/en/catalog/genuine/pick?c=" + i + "&model=Passenger&ssd=%24%2AKwHr18fImIqbmJaKkovBwseci-2egqP9s-Hj9_vi_ZOG6f3xovjzta26_OP67f_90wAAAACxm2NA%24"
    
    # ── Retry up to 3 times ──
    success = False
    for attempt in range(3):
        try:
            print(f"⏳ {i} — attempt {attempt+1}, waiting for page...")
            
            # ---> USE OUR WORKAROUND INSTEAD OF driver.get(url) <---
            driver = handle_cloudflare(driver, url)
            
            # Wait up to 30 seconds for the table to appear
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.CLASS_NAME, "search-result-vin"))
            )
            
            table_div = driver.find_element(By.CLASS_NAME, "search-result-vin")
            rows = table_div.find_elements(By.TAG_NAME, "tr")
            
            for row in rows:
                cells = row.find_elements(By.TAG_NAME, "td")
                if cells:
                    row_data = [cell.text.strip() for cell in cells]
                    try:
                        href = cells[0].find_element(By.TAG_NAME, "a").get_attribute("href")
                    except:
                        href = ""
                    row_data.append(href)
                    row_data.append(i)
                    print(row_data)
                    data.append(row_data)
            
            print(f"✅ {i} done — {len(data)} rows so far")
            success = True
            break  # exit retry loop if successful

        except Exception as e:
            print(f"⚠️ Attempt {attempt+1} failed for {i}: {e}")
            time.sleep(10)  # wait before retrying
    
    if not success:
        print(f"❌ Skipping {i} — all attempts failed")

# Clean up and shape the DataFrame
vehicle = pd.DataFrame(data, columns=["Model", "Market", "Year From", "Year To", "Notes", 'URL', 'Make'])
vehicle['URL2'] = np.where(vehicle['Make'].isna(), vehicle['Notes'], vehicle['URL'])
vehicle['Make'] = np.where(vehicle['Make'].isna(), vehicle['URL'], vehicle['Make'])
vehicle['URL'] = vehicle['URL2']
vehicle.drop(['Notes', 'URL2'], inplace=True, axis=1)
vehicle['URL'] = vehicle['URL'].str.replace('vehicle', 'groups')

# ---> SAVE THE RESULTS AS A CSV INTO THE GITHUB ACTIONS TRACKED FOLDER <---
vehicle.to_csv('output/Scraped_Vehicles.csv', index=False)

In [ ]:
'''vehicle.to_excel("output/vehicle.xlsx", index=False)'''

In [ ]:
'''test'''

In [ ]:
'''driver.get(vehicle['URL'][0])
row_number=[]
for i in range(len(vehicle['URL'])):#len(vehicle['URL'])
    driver.get(vehicle['URL'][i])
    # Click all collapsed expanders to open them
    expanders = driver.find_elements(By.CLASS_NAME, "treegrid-expander-collapsed")
    print(f"Collapsed rows: {len(expanders)}")
    counter=0
    
    for exp in expanders:
        try:
            driver.execute_script("arguments[0].click();", exp)
            time.sleep(0.2)
        except:
            pass
    print('click done:'+str(i))
    time.sleep(1)
    sub_cat=[]
    URL=[]
    # Now grab all links
    links = driver.find_elements(By.XPATH, "//tr[contains(@class, 'treegrid')]//a")
    print('Links taken:'+str(i))
    for link in links:
        text = link.text.strip()
        sub_cat.append(text)
        href = link.get_attribute("href")
        URL.append(href)
        row_number.append(i)
        print('URL taken:'+str(i))
        counter+=1
        print(counter)
Category_URL=vehicle.merge(pd.DataFrame({'rownumber':row_number,'SubCategory':sub_cat,'SubCat_URL':URL}),left_index=True,right_on='rownumber')

part_data = []
for i in range(len(Category_URL['SubCat_URL'])):
    driver.get(Category_URL['SubCat_URL'][i])
   

    # Get category
    category = driver.find_element(
        By.XPATH, "//div[contains(@class,'col-lg-12')]//h2[@style='color: indigo;']"
    ).text.strip().replace("\n", " ")
    
    # Get all subcategory blocks
    blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'unit-header')]/ancestor::div[@class='row']")
    
    for block in blocks:
        # Subcategory name
        try:
            subcategory = block.find_element(
                By.XPATH, ".//div[contains(@class,'unit-header')]//h2"
            ).text.strip()
        except:
            subcategory = ""
    
        # Image URL
        try:
            img = block.find_element(By.XPATH, ".//img[contains(@class,'drag')]")
            img_url = img.get_attribute("src")
        except:
            img_url = ""
    
        # Rows inside this block
        rows = block.find_elements(By.CLASS_NAME, "part-search-tr")
    
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "td")
            if len(cells) >= 5:
                try:
                    number_link = cells[0].find_element(By.TAG_NAME, "a")
                    number      = number_link.text.strip()
                    number_url  = number_link.get_attribute("href")
                except:
                    number      = cells[0].text.strip()
                    number_url  = ""
    
                part_data.append({
                    "category":    category,
                    "subcategory": subcategory,
                    "img_url":     img_url,
                    "number":      number,
                    "number_url":  number_url,
                    "name":        cells[1].text.strip(),
                    "code":        cells[2].text.strip(),
                    "date_range":  cells[3].text.strip(),
                    "options":     cells[4].text.strip(),
                    "quantity":    cells[5].text.strip() if len(cells) > 5 else "",
                    'rownumber':   i
                })



part_data = []

# Get category
category = driver.find_element(
    By.XPATH, "//div[contains(@class,'col-lg-12')]//h2[@style='color: indigo;']"
).text.strip().replace("\n", " ")

# Get all subcategory blocks
blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'unit-header')]/ancestor::div[@class='row']")

for block in blocks:
    # Subcategory name
    try:
        subcategory = block.find_element(
            By.XPATH, ".//div[contains(@class,'unit-header')]//h2"
        ).text.strip()
    except:
        subcategory = ""

    # Image URL
    try:
        img = block.find_element(By.XPATH, ".//img[contains(@class,'drag')]")
        img_url = img.get_attribute("src")
    except:
        img_url = ""

    # Rows inside this block
    rows = block.find_elements(By.CLASS_NAME, "part-search-tr")

    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        if len(cells) >= 5:
            try:
                number_link = cells[0].find_element(By.TAG_NAME, "a")
                number      = number_link.text.strip()
                number_url  = number_link.get_attribute("href")
            except:
                number      = cells[0].text.strip()
                number_url  = ""

            part_data.append({
                "category":    category,
                "subcategory": subcategory,
                "img_url":     img_url,
                "number":      number,
                "number_url":  number_url,
                "name":        cells[1].text.strip(),
                "code":        cells[2].text.strip(),
                "date_range":  cells[3].text.strip(),
                "options":     cells[4].text.strip(),
                "quantity":    cells[5].text.strip() if len(cells) > 5 else ""
            })
Final_data=Category_URL.merge(pd.DataFrame(part_data),left_index=True,right_on='rownumber')
Final_data=Final_data[['Make','Model', 'Market', 'Year From', 'Year To', 'URL',
            'SubCategory', 'SubCat_URL', 'category', 'subcategory',
            'img_url', 'number', 'number_url', 'name', 'code', 'date_range',
            'options', 'quantity']]

driver.get('https://www.autodoc.co.uk/nty/213502B703')

all_data = []

for i in range(1):
    oem = str(Final_data['number'][i])

    driver.get(f"https://www.autodoc.co.uk/car-parts/oem/{oem}")
    time.sleep(3)

    try:
        first_product = driver.find_element(By.CLASS_NAME, "listing-item__name")
        first_product.click()
        time.sleep(3)
    except:
        print(f"No result for {oem}")
        continue

    data = {}
    data["OEM"] = oem

    # Info table
    for item in driver.find_elements(By.CLASS_NAME, "product-description__item"):
        try:
            label = item.find_element(By.CLASS_NAME, "product-description__item-title").text.strip()
            value = item.find_element(By.CLASS_NAME, "product-description__item-value").text.strip()
            data[label] = value
        except:
            pass

    # Images
    img_urls = []
    for item in driver.find_elements(By.CLASS_NAME, "product-gallery__image-list-item"):
        try:
            url = item.find_element(By.TAG_NAME, "img").get_attribute("src")
            if url:
                img_urls.append(url)
        except:
            pass
    data["images"] = img_urls

    # Scroll gradually to trigger lazy loading
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.3);")
    time.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.6);")
    time.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.9);")
    time.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.6);")
    time.sleep(2)

    # Compatibility
    compat = []
    make_blocks = driver.find_elements(By.CLASS_NAME, "product-info-block__item")
    print(f"  Make blocks found: {len(make_blocks)}")

    for make_block in make_blocks:
        try:
            make = make_block.find_element(By.CLASS_NAME, "product-info-block__item-title").get_attribute("innerHTML").strip()
        except:
            make = ""

        driver.execute_script("arguments[0].click();", make_block.find_element(By.CLASS_NAME, "product-info-block__item-title"))
        time.sleep(1)

        model_items = make_block.find_elements(By.CLASS_NAME, "product-info-block__item-list__item")
        for model_item in model_items:
            try:
                model = model_item.find_element(By.CLASS_NAME, "product-info-block__item-list__title").get_attribute("innerHTML").strip()
            except:
                model = ""

            driver.execute_script("arguments[0].click();", model_item.find_element(By.CLASS_NAME, "product-info-block__item-list__title"))
            time.sleep(0.5)

            engines = model_item.find_elements(By.XPATH, ".//ul[contains(@class,'product-info-block__item-sublist')]//li")
            engine_list = [e.get_attribute("innerHTML").strip() for e in engines if e.get_attribute("innerHTML").strip()]

            if engine_list:
                for engine in engine_list:
                    compat.append({"make": make, "model": model, "engine": engine})
            else:
                compat.append({"make": make, "model": model, "engine": ""})

    data["compatibility"] = str(compat)
    all_data.append(data)
    print(f"✅ {oem} done — {len(compat)} compat rows")

autodoc_data_overall = pd.DataFrame(all_data)

autodoc_data_overall=pd.DataFrame([data])



# Parse the string back to list of dicts
autodoc_data_overall['compatibility'] = autodoc_data_overall['compatibility'].apply(ast.literal_eval)

# Explode compatibility into separate rows
df_exploded = autodoc_data_overall.explode('compatibility').reset_index(drop=True)

# Split compatibility dict into separate columns
df_exploded = pd.concat([
    df_exploded.drop(columns=['compatibility']),
    df_exploded['compatibility'].apply(pd.Series)
], axis=1)

Final_data2=Final_data.merge(df_exploded,left_on='number',right_on='OEM')
'''